# Notebook 2 — Category Classification

## Objective
Train and evaluate machine learning models that automatically classify consumer complaints into one of five categories:
- Bank Accounts and Services
- Credit Card Services
- Credit Reporting
- Debt Collection
- Loans

## Pipeline
```
Cleaned text → TF-IDF Vectorization → Train 3 Models → Compare → Select Best → Save
```

## Why text classification works this way
Machine learning models cannot work with raw text — they require numbers. TF-IDF (Term Frequency-Inverse Document Frequency) converts each complaint into a vector of 10,000 numbers, where each number represents how important a specific word or phrase is to that complaint relative to all other complaints.

A word like 'loan' gets a high TF-IDF score in a Loans complaint because it appears frequently there but less so in Credit Card complaints. A word like 'told' gets a near-zero score everywhere because it appears equally across all complaint types — TF-IDF mathematically handles this without manual intervention.

---

In [ ]:
# Cell 1 — Setup & Imports
import os
os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder
import joblib

print('All imports successful')

In [ ]:
# Cell 2 — Load Cleaned Data
df = pd.read_csv('data/cleaned_tickets.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nCategory distribution:\n{df["product_5"].value_counts()}')
df.head(3)

---
## Step 1 — Feature Engineering with TF-IDF

### TF-IDF parameter choices

| Parameter | Value | Reason |
|---|---|---|
| `max_features` | 10,000 | Captures more vocabulary than 5,000; tested and confirmed improvement |
| `ngram_range` | (1, 2) | Captures both single words AND two-word phrases like 'credit card', 'debt collector' |
| `min_df` | 2 | Ignores terms appearing in fewer than 2 documents — removes typos and rare noise |
| `sublinear_tf` | True | Applies log normalization to term frequency — prevents very frequent words from dominating |

### Why bigrams matter
'credit' and 'card' separately appear in many categories. 'credit card' as a bigram is much more specific to Credit Card Services. Enabling `ngram_range=(1,2)` lets TF-IDF capture these compound signals.

---

In [ ]:
# Cell 3 — Prepare Features & Labels
df = df.dropna(subset=['cleaned_text'])
df['cleaned_text'] = df['cleaned_text'].astype(str)

X = df['cleaned_text']
y = df['product_5']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f'Features (X): {X.shape[0]} samples')
print(f'Label classes: {list(le.classes_)}')
print(f'Encoded labels sample: {y_encoded[:5]}')

In [ ]:
# Cell 4 — TF-IDF Vectorization
tfidf = TfidfVectorizer(
    max_features=10000,  # increased from 5000 after testing — +0.26% accuracy
    ngram_range=(1, 2),  # unigrams and bigrams
    min_df=2,
    sublinear_tf=True
)

X_tfidf = tfidf.fit_transform(X)

print(f'TF-IDF matrix shape: {X_tfidf.shape}')
print(f'  {X_tfidf.shape[0]} complaints x {X_tfidf.shape[1]} features')
print(f'\nSample features: {tfidf.get_feature_names_out()[:20]}')

In [ ]:
# Cell 5 — Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')
print(f'\nClass distribution in training set:')
unique, counts = np.unique(y_train, return_counts=True)
for label, count in zip(le.inverse_transform(unique), counts):
    print(f'  {label}: {count}')

---
## Step 2 — Model Training

Three models are trained and compared. Each represents a different approach to classification:

### Logistic Regression
Assigns a learned weight to each TF-IDF feature for each category. Classification is a weighted sum — the category with the highest score wins. Weights are learned via gradient descent on the training data. Fast, interpretable, strong baseline for text classification.

### Random Forest
Builds 200 decision trees, each trained on a random subset of data and features. Each tree votes on the category and the majority wins. Robust to overfitting through randomness. Generally underperforms linear models on text classification but included for comparison.

### LinearSVC
Finds the optimal hyperplane separating each category in the 10,000-dimensional TF-IDF space. Specifically designed for high-dimensional sparse data — exactly what TF-IDF produces. Wrapped in `CalibratedClassifierCV` to enable probability estimates for the inference demo.

---

In [ ]:
# Cell 6 — Train Logistic Regression
lr_model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver='lbfgs',
    random_state=42
)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
lr_acc = accuracy_score(y_test, lr_preds)
print(f'Logistic Regression Accuracy: {lr_acc:.4f} ({lr_acc*100:.2f}%)')

In [ ]:
# Cell 7 — Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)
print(f'Random Forest Accuracy: {rf_acc:.4f} ({rf_acc*100:.2f}%)')

In [ ]:
# Cell 7b — Train LinearSVC
svm_model = CalibratedClassifierCV(
    LinearSVC(C=1.0, max_iter=2000, random_state=42)
)
svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_test)
svm_acc = accuracy_score(y_test, svm_preds)
print(f'LinearSVC Accuracy: {svm_acc:.4f} ({svm_acc*100:.2f}%)')

In [ ]:
# Cell 8 — Compare All Three Models
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'LinearSVC'],
    'Accuracy': [lr_acc, rf_acc, svm_acc]
}).sort_values('Accuracy', ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#e74c3c' if v == results['Accuracy'].max()
          else 'steelblue' for v in results['Accuracy']]
bars = ax.barh(results['Model'], results['Accuracy'],
               color=colors, edgecolor='white')
ax.set_xlim(0.75, 0.90)
ax.set_xlabel('Accuracy')
ax.set_title('Category Classification — Model Comparison', fontweight='bold')
for bar, val in zip(bars, results['Accuracy']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val*100:.2f}%', va='center', fontsize=11)
plt.tight_layout()
plt.savefig('data/category_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(results.to_string(index=False))

---
## Step 3 — Best Model Selection & Evaluation

**Winner: Logistic Regression**

Logistic Regression outperforms both LinearSVC and Random Forest on this task. This aligns with the general principle that for text classification with TF-IDF features, linear models tend to perform well because the feature space is high-dimensional and sparse — exactly the conditions where linear decision boundaries are most effective.

Random Forest underperforms because decision trees struggle with high-dimensional sparse data — most features are zero for any given document, making tree splitting decisions unreliable.

---

In [ ]:
# Cell 9 — Best Model Selection & Classification Report
model_scores = {
    'Logistic Regression': (lr_acc, lr_model, lr_preds),
    'LinearSVC'          : (svm_acc, svm_model, svm_preds),
    'Random Forest'      : (rf_acc, rf_model, rf_preds)
}

best_name = max(model_scores, key=lambda x: model_scores[x][0])
best_acc, best_model, best_preds = model_scores[best_name]

print(f'Best model: {best_name}\n')
print('=== Classification Report ===')
print(classification_report(
    y_test, best_preds,
    target_names=le.classes_
))

In [ ]:
# Cell 10 — Confusion Matrix
cm = confusion_matrix(y_test, best_preds)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=13, fontweight='bold', pad=12)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('data/category_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved.')

In [ ]:
# Cell 11 — Per-Class Performance Chart
report = classification_report(
    y_test, best_preds,
    target_names=le.classes_,
    output_dict=True
)
report_df = pd.DataFrame(report).T.iloc[:-3]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(report_df))
width = 0.25
ax.bar(x - width, report_df['precision'], width, label='Precision', color='steelblue')
ax.bar(x,         report_df['recall'],    width, label='Recall',    color='seagreen')
ax.bar(x + width, report_df['f1-score'],  width, label='F1-Score',  color='coral')
ax.set_xticks(x)
ax.set_xticklabels(report_df.index, rotation=20, ha='right')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title(f'Per-Class Performance — {best_name}', fontweight='bold')
ax.legend()
ax.axhline(0.8, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
plt.tight_layout()
plt.savefig('data/category_per_class_performance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 4 — Hyperparameter Tuning

GridSearchCV systematically tries every combination of specified hyperparameters using 5-fold cross-validation and identifies the optimal configuration.

### Parameters searched
- **C** (regularization strength): 0.1, 1.0, 5.0, 10.0, 50.0
- **solver**: lbfgs, saga

### Result
GridSearchCV confirmed C=1.0 with saga solver as optimal. This tells us the model has reached its performance ceiling with the current feature set. The 10,000 feature TF-IDF vocabulary is the primary driver of accuracy, not the regularization parameter.

**Key insight:** When GridSearchCV confirms default parameters are optimal, it means the model architecture is well-suited to the task. Further accuracy improvements would require richer features (e.g. transformer embeddings) rather than parameter tuning.

---

In [ ]:
# Cell 9b — Hyperparameter Tuning with GridSearchCV
print('Running GridSearchCV — this may take 2-3 minutes...')

param_grid = {
    'C': [0.1, 1.0, 5.0, 10.0, 50.0],
    'solver': ['lbfgs', 'saga']
}

grid_search = GridSearchCV(
    LogisticRegression(max_iter=2000, random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV accuracy: {grid_search.best_score_*100:.2f}%')

tuned_preds = grid_search.best_estimator_.predict(X_test)
tuned_acc = accuracy_score(y_test, tuned_preds)
print(f'Test accuracy with tuned model: {tuned_acc*100:.2f}%')
print(f'Improvement over baseline: +{(tuned_acc - lr_acc)*100:.2f}%')

# Update best model if tuned version is better
if tuned_acc > best_acc:
    best_model = grid_search.best_estimator_
    best_preds = tuned_preds
    best_acc   = tuned_acc
    best_name  = 'Logistic Regression (Tuned)'
    print('\nTuned model replaces baseline as best model.')
else:
    print('\nBaseline model retained — tuning did not improve results.')

In [ ]:
# Cell 12 — Save Model & Vectorizer
os.makedirs('models', exist_ok=True)

joblib.dump(best_model, 'models/category_model.pkl')
joblib.dump(tfidf,      'models/tfidf_vectorizer.pkl')
joblib.dump(le,         'models/category_label_encoder.pkl')

print('Models saved:')
print('  models/category_model.pkl')
print('  models/tfidf_vectorizer.pkl')
print('  models/category_label_encoder.pkl')
print(f'\nBest model  : {best_name}')
print(f'Accuracy    : {best_acc*100:.2f}%')
print(f'Classes     : {list(le.classes_)}')

In [ ]:
# Cell 13 — Quick Inference Test
sample_tickets = [
    "There is an error on my credit report from Equifax that does not belong to me",
    "A debt collector keeps calling me about a debt I already paid off last year",
    "I was charged an overdraft fee even though I had sufficient funds in my account",
    "My mortgage payment was applied incorrectly and now shows as late",
    "Someone opened a credit card in my name without my permission"
]

sample_tfidf = tfidf.transform(sample_tickets)
sample_preds = best_model.predict(sample_tfidf)
sample_labels = le.inverse_transform(sample_preds)

print('=== Sample Predictions ===\n')
for ticket, label in zip(sample_tickets, sample_labels):
    print(f'Ticket  : {ticket}')
    print(f'Category: {label}')
    print()

---
## Notebook Summary

| Model | Accuracy | Notes |
|---|---|---|
| Logistic Regression | 85.02% | Best — linear models excel on TF-IDF features |
| LinearSVC | 84.76% | Close second — also strong on sparse high-dim data |
| Random Forest | 82.92% | Weakest — struggles with high-dimensional sparse vectors |

### Key findings
- All models exceed 80% accuracy — strong performance on a 5-class real-world task
- The main confusion is between Credit Reporting and Debt Collection — semantically the most similar categories since debt collectors frequently appear in credit report complaints
- GridSearchCV confirmed default parameters are optimal — the bottleneck is feature representation, not model tuning
- Increasing TF-IDF vocabulary from 5,000 to 10,000 features provided a +0.26% accuracy gain

**Next:** Notebook 3 builds the priority prediction model with sentiment-enhanced label engineering.

---